In [ ]:
import pandas as pd
import numpy as np
import shutil, os
from deltalake import DeltaTable, write_deltalake

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 140)

DATA_DIR = "../data"
DELTA_DIR = "../delta"          

shutil.rmtree(DELTA_DIR, ignore_errors=True)
os.makedirs(DELTA_DIR, exist_ok=True)

print("deltalake / pandas environment ready.")


deltalake / pandas environment ready.


In [ ]:
master_raw = pd.read_csv(f"{DATA_DIR}/customer_master.csv")
print("Rows loaded from CSV:", len(master_raw))
master_raw.head()


Rows loaded from CSV: 660


,customer_id,customer_name,segment,country,city,state,postal_code,region,first_order_date,total_sales,total_profit,total_orders
0,KC-16255,Karen Carlisle,Corporate,United States,Macon,Georgia,31204.0,South,2014-09-08,2120.95,846.12,6
1,MC-17605,Matt Connell,Corporate,United States,Lakeland,Florida,33801.0,South,2014-03-19,2258.19,195.45,8
2,ST-20530,Shui Tom,Consumer,United States,Houston,Texas,77095.0,Central,2014-02-14,433.34,84.44,7
3,KC-16540,Kelly Collister,Consumer,United States,Seattle,Washington,98115.0,West,2015-11-02,3908.26,709.42,4
4,BS-11755,Bruce Stewart,Consumer,United States,Cleveland,Ohio,44105.0,East,2014-05-18,2562.38,-113.30,7


In [ ]:
write_deltalake(f"{DELTA_DIR}/customer_raw", master_raw, mode="overwrite")

dt_raw = DeltaTable(f"{DELTA_DIR}/customer_raw")
print("Delta table created at:", f"{DELTA_DIR}/customer_raw")
print("Delta table version :", dt_raw.version())
print("Delta table schema  :")
print(dt_raw.schema())


Delta table created at: ../delta/customer_raw
Delta table version : 0
Delta table schema  :
Schema([Field(customer_id, PrimitiveType("string"), nullable=True), Field(customer_name, PrimitiveType("string"), nullable=True), Field(segment, PrimitiveType("string"), nullable=True), Field(country, PrimitiveType("string"), nullable=True), Field(city, PrimitiveType("string"), nullable=True), Field(state, PrimitiveType("string"), nullable=True), Field(postal_code, PrimitiveType("double"), nullable=True), Field(region, PrimitiveType("string"), nullable=True), Field(first_order_date, PrimitiveType("string"), nullable=True), Field(total_sales, PrimitiveType("double"), nullable=True), Field(total_profit, PrimitiveType("double"), nullable=True), Field(total_orders, PrimitiveType("long"), nullable=True)])


In [ ]:
df_check = dt_raw.to_pandas()
print("Rows in Delta table:", len(df_check))
df_check.head()


Rows in Delta table: 660


,customer_id,customer_name,segment,country,city,state,postal_code,region,first_order_date,total_sales,total_profit,total_orders
0,KC-16255,Karen Carlisle,Corporate,United States,Macon,Georgia,31204.0,South,2014-09-08,2120.95,846.12,6
1,MC-17605,Matt Connell,Corporate,United States,Lakeland,Florida,33801.0,South,2014-03-19,2258.19,195.45,8
2,ST-20530,Shui Tom,Consumer,United States,Houston,Texas,77095.0,Central,2014-02-14,433.34,84.44,7
3,KC-16540,Kelly Collister,Consumer,United States,Seattle,Washington,98115.0,West,2015-11-02,3908.26,709.42,4
4,BS-11755,Bruce Stewart,Consumer,United States,Cleveland,Ohio,44105.0,East,2014-05-18,2562.38,-113.30,7


In [ ]:
raw = dt_raw.to_pandas()

print("=== Null counts BEFORE cleaning ===")
print(raw.isnull().sum()[raw.isnull().sum() > 0])
print()
print("Exact duplicate rows:", raw.duplicated().sum())
print("Duplicate customer_id count:", raw['customer_id'].duplicated().sum())


=== Null counts BEFORE cleaning ===
segment        5
city           5
postal_code    5
dtype: int64

Exact duplicate rows: 10
Duplicate customer_id count: 10


In [ ]:
clean = raw.copy()

before = len(clean)
clean = clean.drop_duplicates()
print(f"Removed {before - len(clean)} exact duplicate rows")

clean['segment'] = clean['segment'].fillna('Unknown')
clean['city'] = clean['city'].fillna('Unknown')
clean['postal_code'] = clean['postal_code'].fillna(clean['postal_code'].median())
clean['postal_code'] = clean['postal_code'].astype(int)


clean['completeness'] = clean.notnull().sum(axis=1)
clean = (clean.sort_values('completeness', ascending=False)
               .drop_duplicates(subset='customer_id', keep='first')
               .drop(columns='completeness')
               .reset_index(drop=True))

print()
print("=== Null counts AFTER cleaning ===")
print(clean.isnull().sum().sum(), "total nulls remaining")
print("Rows after cleaning:", len(clean), " | Unique customer_id:", clean['customer_id'].nunique())


Removed 10 exact duplicate rows

=== Null counts AFTER cleaning ===
0 total nulls remaining
Rows after cleaning: 650  | Unique customer_id: 650


In [ ]:
write_deltalake(f"{DELTA_DIR}/customer_silver", clean, mode="overwrite")

dt_silver = DeltaTable(f"{DELTA_DIR}/customer_silver")
print("customer_silver Delta table version:", dt_silver.version())
clean.head()


customer_silver Delta table version: 0


,customer_id,customer_name,segment,country,city,state,postal_code,region,first_order_date,total_sales,total_profit,total_orders
0,KC-16255,Karen Carlisle,Corporate,United States,Macon,Georgia,31204,South,2014-09-08,2120.95,846.12,6
1,MC-17605,Matt Connell,Corporate,United States,Lakeland,Florida,33801,South,2014-03-19,2258.19,195.45,8
2,ST-20530,Shui Tom,Consumer,United States,Houston,Texas,77095,Central,2014-02-14,433.34,84.44,7
3,KC-16540,Kelly Collister,Consumer,United States,Seattle,Washington,98115,West,2015-11-02,3908.26,709.42,4
4,BS-11755,Bruce Stewart,Consumer,United States,Cleveland,Ohio,44105,East,2014-05-18,2562.38,-113.30,7


In [ ]:
incremental = pd.read_csv(f"{DATA_DIR}/customer_incremental.csv")
print("Incremental batch rows:", len(incremental))

existing_ids = set(clean['customer_id'])
incoming_ids = set(incremental['customer_id'])

n_new = len(incoming_ids - existing_ids)
n_updates = len(incoming_ids & existing_ids)
print(f"  -> brand-new customers (inserts): {n_new}")
print(f"  -> existing customers  (updates): {n_updates}")

incremental.head()


Incremental batch rows: 223
  -> brand-new customers (inserts): 143
  -> existing customers  (updates): 80


,customer_id,customer_name,segment,country,city,state,postal_code,region,first_order_date,total_sales,total_profit,total_orders
0,MC-17590,Matt Collister,Corporate,United States,San Marcos,Texas,78666,Central,2014-05-20,2426.07,288.98,6
1,CD-11920,Carlos Daly,Consumer,United States,New York City,New York,10024,East,2014-03-24,2033.97,426.66,5
2,JB-16045,Julia Barnett,Home Office,United States,Chula Vista,California,91911,West,2015-05-03,2518.12,201.52,5
3,AD-10180,Alan Dominguez,Consumer,United States,Philadelphia,Pennsylvania,19134,East,2014-11-19,7836.84,2598.79,11
4,CC-12550,Clay Cheatham,Consumer,United States,San Francisco,California,94122,West,2017-06-15,113.83,33.87,3


In [ ]:
write_deltalake(f"{DELTA_DIR}/customer_scd1", clean, mode="overwrite")
dt_scd1 = DeltaTable(f"{DELTA_DIR}/customer_scd1")

rows_before_scd1 = len(dt_scd1.to_pandas())
print("customer_scd1 rows BEFORE merge:", rows_before_scd1)

(
    dt_scd1.merge(
        source=incremental,
        predicate="target.customer_id = source.customer_id",
        source_alias="source",
        target_alias="target",
    )
    .when_matched_update_all()
    .when_not_matched_insert_all()
    .execute()
)

dt_scd1 = DeltaTable(f"{DELTA_DIR}/customer_scd1")   # reload to see the new version
rows_after_scd1 = len(dt_scd1.to_pandas())
print("customer_scd1 rows AFTER  merge:", rows_after_scd1)
print("New Delta version:", dt_scd1.version())


customer_scd1 rows BEFORE merge: 650


customer_scd1 rows AFTER  merge: 793
New Delta version: 1


In [ ]:
sample_updated_id = incremental[incremental['customer_id'].isin(clean['customer_id'])]['customer_id'].iloc[0]
print("BEFORE (silver):")
display(clean[clean['customer_id'] == sample_updated_id])
print("AFTER (scd1, post-merge) -- old values are gone, only current state remains:")
display(dt_scd1.to_pandas().query("customer_id == @sample_updated_id"))


BEFORE (silver):


,customer_id,customer_name,segment,country,city,state,postal_code,region,first_order_date,total_sales,total_profit,total_orders
624,AD-10180,Alan Dominguez,Home Office,United States,Philadelphia,Pennsylvania,19134,East,2014-11-19,6106.88,1869.93,8


AFTER (scd1, post-merge) -- old values are gone, only current state remains:


,customer_id,customer_name,segment,country,city,state,postal_code,region,first_order_date,total_sales,total_profit,total_orders
217,AD-10180,Alan Dominguez,Consumer,United States,Philadelphia,Pennsylvania,19134,East,2014-11-19,7836.84,2598.79,11


In [ ]:
import pyarrow as pa

scd2_init = clean.copy()
scd2_init['effective_date'] = '2026-01-01'
scd2_init['is_current'] = True

table_init = pa.Table.from_pandas(scd2_init, preserve_index=False)
table_init = table_init.append_column('end_date', pa.array([None] * len(scd2_init), type=pa.string()))

write_deltalake(f"{DELTA_DIR}/customer_scd2", table_init, mode="overwrite")
dt_scd2 = DeltaTable(f"{DELTA_DIR}/customer_scd2")
print("customer_scd2 initial rows:", len(dt_scd2.to_pandas()), "| version:", dt_scd2.version())


customer_scd2 initial rows: 650 | version: 0


In [ ]:
TRACKED_COLS = ['segment', 'city', 'total_sales', 'total_profit', 'total_orders']
TODAY = '2026-07-31'

current = dt_scd2.to_pandas()
current_active = current[current['is_current'] == True]

merged_check = incremental.merge(
    current_active[['customer_id'] + TRACKED_COLS],
    on='customer_id', how='left', suffixes=('', '_old'), indicator=True
)

is_new       = merged_check['_merge'] == 'left_only'
is_changed   = (merged_check['_merge'] == 'both') & (
    (merged_check[TRACKED_COLS].values != merged_check[[c + '_old' for c in TRACKED_COLS]].values).any(axis=1)
)

new_customers_scd2     = incremental[is_new.values].copy()
changed_customers_scd2 = incremental[is_changed.values].copy()

print("Brand-new customers to insert :", len(new_customers_scd2))
print("Changed customers (new version):", len(changed_customers_scd2))


Brand-new customers to insert : 143
Changed customers (new version): 80


In [ ]:
expiring_keys = changed_customers_scd2[['customer_id']].copy()

(
    dt_scd2.merge(
        source=expiring_keys,
        predicate="target.customer_id = source.customer_id AND target.is_current = true",
        source_alias="source",
        target_alias="target",
    )
    .when_matched_update(updates={"is_current": "false", "end_date": f"'{TODAY}'"})
    .execute()
)

dt_scd2 = DeltaTable(f"{DELTA_DIR}/customer_scd2")
print("Rows expired. Current Delta version:", dt_scd2.version())
print(dt_scd2.to_pandas().query("is_current == False").shape[0], "row(s) now marked historical")


Rows expired. Current Delta version: 1
80 row(s) now marked historical


In [ ]:
new_versions = pd.concat([new_customers_scd2, changed_customers_scd2], ignore_index=True)
new_versions['effective_date'] = TODAY
new_versions['is_current'] = True

table_new = pa.Table.from_pandas(new_versions, preserve_index=False)
table_new = table_new.append_column('end_date', pa.array([None] * len(new_versions), type=pa.string()))

write_deltalake(f"{DELTA_DIR}/customer_scd2", table_new, mode="append")

dt_scd2 = DeltaTable(f"{DELTA_DIR}/customer_scd2")
print("Rows appended:", len(new_versions))
print("customer_scd2 total rows now:", len(dt_scd2.to_pandas()), "| version:", dt_scd2.version())


Rows appended: 223
customer_scd2 total rows now: 873 | version: 2


In [ ]:
example_id = changed_customers_scd2['customer_id'].iloc[0]
history_view = dt_scd2.to_pandas().query("customer_id == @example_id").sort_values('effective_date')
history_view[['customer_id', 'segment', 'total_sales', 'total_orders', 'effective_date', 'end_date', 'is_current']]


,customer_id,segment,total_sales,total_orders,effective_date,end_date,is_current
297,AD-10180,Home Office,6106.88,8,2026-01-01,2026-07-31,False
143,AD-10180,Consumer,7836.84,11,2026-07-31,NaN,True


In [16]:
print("### SCD1 validation ###")
scd1_final = dt_scd1.to_pandas()
expected = rows_before_scd1 + n_new
print(f"expected rows (initial {rows_before_scd1} + new {n_new}) = {expected}")
print(f"actual rows in customer_scd1                              = {len(scd1_final)}")
assert len(scd1_final) == expected, "Row count mismatch!"
print("Row-count check:  PASSED\n")

dup_ids = scd1_final['customer_id'].duplicated().sum()
print(f"Duplicate customer_id in customer_scd1: {dup_ids}")
assert dup_ids == 0
print("No-duplicates check: PASSED")


### SCD1 validation ###
expected rows (initial 650 + new 143) = 793
actual rows in customer_scd1                              = 793
Row-count check:  PASSED

Duplicate customer_id in customer_scd1: 0
No-duplicates check: PASSED


In [17]:
print("### SCD2 validation ###")
scd2_final = dt_scd2.to_pandas()
current_rows = scd2_final[scd2_final['is_current'] == True]
historical_rows = scd2_final[scd2_final['is_current'] == False]

print("Total rows (all versions):", len(scd2_final))
print("Current rows              :", len(current_rows))
print("Historical rows           :", len(historical_rows))

dup_current_ids = current_rows['customer_id'].duplicated().sum()
print(f"Duplicate customer_id among CURRENT rows: {dup_current_ids}")
assert dup_current_ids == 0
print("No-duplicates-among-current check: PASSED")

assert len(historical_rows) == len(changed_customers_scd2)
print(f"History rows == changed customers ({len(changed_customers_scd2)}): PASSED")


### SCD2 validation ###
Total rows (all versions): 873
Current rows              : 793
Historical rows           : 80
Duplicate customer_id among CURRENT rows: 0
No-duplicates-among-current check: PASSED
History rows == changed customers (80): PASSED


In [18]:
print("### Delta transaction log / version history ###\n")
print("-- customer_raw --")
display(DeltaTable(f"{DELTA_DIR}/customer_raw").history())
print("\n-- customer_scd1 --")
display(dt_scd1.history())
print("\n-- customer_scd2 --")
display(dt_scd2.history())


### Delta transaction log / version history ###

-- customer_raw --


[{'timestamp': 1785522389155,
  'operation': 'WRITE',
  'operationParameters': {'mode': 'Overwrite'},
  'engineInfo': 'delta-rs:py-1.6.2',
  'clientVersion': 'delta-rs.py-1.6.2',
  'operationMetrics': {'execution_time_ms': 10,
   'num_added_files': 1,
   'num_added_rows': 660,
   'num_partitions': 0,
   'num_removed_files': 0},
  'version': 0}]


-- customer_scd1 --


[{'timestamp': 1785522389398,
  'operation': 'MERGE',
  'operationParameters': {'mergePredicate': 'target.customer_id = source.customer_id',
   'matchedPredicates': '[{"actionType":"update"}]',
   'notMatchedBySourcePredicates': '[]',
   'notMatchedPredicates': '[{"actionType":"insert"}]'},
  'readVersion': 0,
  'engineInfo': 'delta-rs:py-1.6.2',
  'clientVersion': 'delta-rs.py-1.6.2',
  'operationMetrics': {'execution_time_ms': 42,
   'num_output_rows': 793,
   'num_source_rows': 223,
   'num_target_files_added': 1,
   'num_target_files_removed': 1,
   'num_target_files_scanned': 1,
   'num_target_files_skipped_during_scan': 0,
   'num_target_rows_copied': 570,
   'num_target_rows_deleted': 0,
   'num_target_rows_inserted': 143,
   'num_target_rows_updated': 80,
   'rewrite_time_ms': 2,
   'scan_time_ms': 10},
  'version': 1},
 {'timestamp': 1785522389342,
  'operation': 'WRITE',
  'operationParameters': {'mode': 'Overwrite'},
  'engineInfo': 'delta-rs:py-1.6.2',
  'clientVersion': 'd


-- customer_scd2 --


[{'timestamp': 1785522389616,
  'operation': 'WRITE',
  'operationParameters': {'mode': 'Append'},
  'engineInfo': 'delta-rs:py-1.6.2',
  'clientVersion': 'delta-rs.py-1.6.2',
  'operationMetrics': {'execution_time_ms': 3,
   'num_added_files': 1,
   'num_added_rows': 223,
   'num_partitions': 0,
   'num_removed_files': 0},
  'version': 2},
 {'timestamp': 1785522389579,
  'operation': 'MERGE',
  'operationParameters': {'mergePredicate': 'target.customer_id = source.customer_id AND target.is_current',
   'notMatchedBySourcePredicates': '[]',
   'predicate': 'is_current',
   'matchedPredicates': '[{"actionType":"update"}]',
   'notMatchedPredicates': '[]'},
  'readVersion': 0,
  'engineInfo': 'delta-rs:py-1.6.2',
  'clientVersion': 'delta-rs.py-1.6.2',
  'operationMetrics': {'execution_time_ms': 26,
   'num_output_rows': 650,
   'num_source_rows': 80,
   'num_target_files_added': 1,
   'num_target_files_removed': 1,
   'num_target_files_scanned': 1,
   'num_target_files_skipped_during_sc

In [19]:
final_df = dt_scd1.to_pandas().sort_values('customer_id').reset_index(drop=True)
print("Final customer_scd1 table --", len(final_df), "rows,", final_df['customer_id'].nunique(), "unique customers")
final_df.head(10)


Final customer_scd1 table -- 793 rows, 793 unique customers


,customer_id,customer_name,segment,country,city,state,postal_code,region,first_order_date,total_sales,total_profit,total_orders
0,AA-10315,Alex Avila,Consumer,United States,San Francisco,California,94122,West,2014-03-31,5563.56,-362.88,5
1,AA-10375,Allen Armold,Consumer,United States,Los Angeles,California,90008,West,2014-04-21,1056.39,277.38,9
2,AA-10480,Andrew Allen,Consumer,United States,Middletown,Connecticut,6457,East,2014-05-04,1790.51,435.83,4
3,AA-10645,Anna Andreadi,Consumer,United States,Chester,Pennsylvania,19013,East,2014-06-22,5086.93,857.80,6
4,AB-10015,Aaron Bergman,Consumer,United States,Arlington,Texas,76017,Central,2014-02-18,886.16,129.35,3
5,AB-10060,Adam Bellavance,Home Office,United States,Des Moines,Washington,98198,West,2015-09-18,7755.62,2054.59,8
6,AB-10105,Adrian Barton,Consumer,United States,Indianapolis,Indiana,46203,Central,2014-12-20,14473.57,5444.81,10
7,AB-10150,Aimee Bixby,Consumer,United States,Yonkers,New York,10701,East,2014-03-05,966.71,313.66,5
8,AB-10165,Alan Barnes,Consumer,United States,Decatur,Illinois,62521,Central,2014-11-16,1113.84,220.81,8
9,AB-10255,Alejandro Ballentine,Home Office,United States,Los Angeles,California,90008,West,2014-07-22,914.53,264.57,9


In [ ]:
final_df.describe(include='all').T.head(12)

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
customer_id,793,793,AA-10315,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
customer_name,793,793,Alex Avila,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
segment,793,4,Consumer,375,NaN,NaN,NaN,NaN,NaN,NaN,NaN
country,793,1,United States,793,NaN,NaN,NaN,NaN,NaN,NaN,NaN
city,793,245,Los Angeles,59,NaN,NaN,NaN,NaN,NaN,NaN,NaN
state,793,43,California,156,NaN,NaN,NaN,NaN,NaN,NaN,NaN
postal_code,793.0,NaN,NaN,NaN,56064.213115,31364.872284,1841.0,28027.0,60201.0,90004.0,99301.0
region,793,4,West,252,NaN,NaN,NaN,NaN,NaN,NaN,NaN
first_order_date,793,446,2014-09-08,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN
total_sales,793.0,NaN,NaN,NaN,2952.299344,2686.450506,4.83,1171.81,2280.58,3887.83,25043.05


In [21]:
summary = pd.DataFrame({
    "Metric": [
        "Raw rows loaded (bronze)",
        "Duplicate rows removed",
        "Nulls remaining after cleaning",
        "Rows after cleaning (silver)",
        "Incremental batch size",
        "New customers inserted",
        "Existing customers updated",
        "Final row count - SCD1 (current-state)",
        "Final row count - SCD2 (all versions, incl. history)",
        "Historical (superseded) rows kept by SCD2",
    ],
    "Value": [
        len(master_raw),
        int(before - len(clean.drop_duplicates()) + 0) if False else (raw.duplicated().sum() + raw['customer_id'].duplicated().sum()),
        clean.isnull().sum().sum(),
        len(clean),
        len(incremental),
        n_new,
        n_updates,
        len(scd1_final),
        len(scd2_final),
        len(historical_rows),
    ]
})
summary


,Metric,Value
0,Raw rows loaded (bronze),660
1,Duplicate rows removed,20
2,Nulls remaining after cleaning,0
3,Rows after cleaning (silver),650
4,Incremental batch size,223
5,New customers inserted,143
6,Existing customers updated,80
7,Final row count - SCD1 (current-state),793
8,"Final row count - SCD2 (all versions, incl. hi...",873
9,Historical (superseded) rows kept by SCD2,80
